# Compose a Granite Switch checkpoint

**Duration:** ~10-20 min (first run, mostly download)

This notebook shows how to compose a Granite Switch checkpoint yourself: combine a base Granite model with one or more LoRA adapter libraries into a single artifact you can serve with vLLM and drive from mellea. Sibling tutorials ([`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb), [`02_govt_rag_pipeline.ipynb`](02_govt_rag_pipeline.ipynb)) **consume** such a checkpoint - this one **produces** one.

**What you'll learn:**
- How the composer pulls base weights and LoRA libraries into one checkpoint
- How to preview library contents with `--list-adapters` before committing to a build
- How to trim the checkpoint with `--include-adapters` / `--exclude-adapters` / `--technology-filter`
- How to point vLLM and mellea at the result and confirm the embedded adapters are live

**Adapters used:** this notebook builds a checkpoint that embeds all three IBM granitelib libraries - [Core](https://huggingface.co/ibm-granite/granitelib-core-r1.0), [RAG](https://huggingface.co/ibm-granite/granitelib-rag-r1.0), and [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) - into a single base Granite model, then verifies the result by invoking one RAG intrinsic (`rewrite_question`).

section 2 and section 3 do the actual work; section 4 is a recipe book of selection flags (pre-commented so re-running the notebook doesn't rebuild multiple checkpoints). For the canonical CLI reference see [`src/granite_switch/composer/README.md`](../../src/granite_switch/composer/README.md).

## Prerequisites

1. **Install the composer extras** (CPU is fine for section 1-section 4; section 6 needs a GPU):
   


In [ ]:
!pip install mellea "granite-switch[compose,vllm]"


2. **Hugging Face login** - required if any adapter library or the base model is gated:
   


In [ ]:
!huggingface-cli login


3. **Disk and bandwidth.** A full build of all three libraries downloads the base model (~6 GB for `granite-4.1-3b` bf16) plus adapters, and writes the composed checkpoint to disk. Plan for ~20 GB free space and several minutes on a fast connection. Subsequent runs hit the HF cache.
4. **For section 5-section 6 only - vLLM.** Install `vllm` (`pip install "granite-switch[vllm]"`) and have a GPU available. You won't need it until section 5.


## 1 * Configuration

Edit these if you want a different base model, different libraries, or a different output directory.

In [ ]:
# Imports
from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.context import ChatContext

BASE_MODEL   = "ibm-granite/granite-4.1-3b"
OUTPUT_DIR   = "./granite-switch-tutorials"

RAG_LIB      = "ibm-granite/granitelib-rag-r1.0"       # query_rewrite, answerability, citations, ...
CORE_LIB     = "ibm-granite/granitelib-core-r1.0"      # requirement-check, uncertainty, context-attribution
GUARDIAN_LIB = "ibm-granite/granitelib-guardian-r1.0"  # guardian-core, policy-guardrails, factuality-*

print(f"base:    {BASE_MODEL}")
print(f"output:  {OUTPUT_DIR}")
print(f"libs:    {RAG_LIB}\n         {CORE_LIB}\n         {GUARDIAN_LIB}")


## 2 * Preview what's available - `--list-adapters`

Ask the composer what each library contains. `--list-adapters` prints the adapter inventory and exits without writing anything - useful for deciding what to include before committing to a full build. When both `alora` and `lora` flavors exist for the same adapter, the composer prefers `alora` by default.

> **Note:** This step is optional - it only previews the adapters in each library and doesn't affect the build in section 3. If the cell fails (e.g. `unrecognized arguments: --list-adapters`) it usually means the `granite_switch` package on your notebook kernel's Python path is older than this repo. You can safely skip ahead to section 3; the compose step doesn't need `--list-adapters`. To fix it, reinstall the package in the kernel's environment (`pip install "granite-switch[compose]"`) and restart the kernel.

In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
  --list-adapters

## 3 * Compose the model

Pull all adapters from all three libraries and embed them into the base model. Takes a few minutes and downloads ~15 GB (base model + adapters) on the first run; subsequent runs hit the HF cache.

In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
  --output {OUTPUT_DIR}

Two files in the output directory are worth looking at. **`BUILD.md`** is a human-readable summary - the adapter table in it tells you the control token (e.g. `<|answerability|>`) that mellea will route intrinsic calls through. **`adapter_index.json`** is the same mapping in machine-readable form, used at inference time.

In [ ]:
!ls {OUTPUT_DIR}
print("\n--- first 60 lines of BUILD.md ---")
!head -60 {OUTPUT_DIR}/BUILD.md

## 4 * Selecting which adapters to include

By default the composer embeds every adapter it finds in the libraries you point it at. That's a reasonable place to start, but for production you'll often want a leaner checkpoint: fewer embedded adapters means a smaller safetensors file, lower VRAM at serve time, and a faster cold start.

These flags narrow the set:

| Flag | What it does |
|---|---|
| `--include-adapters` | Keep only adapters matching these names or [fnmatch globs](https://docs.python.org/3/library/fnmatch.html) (e.g. `'query_*'`). |
| `--exclude-adapters` | Drop adapters matching these names/globs. Applied *after* `--include-adapters`. |
| `--technology-filter` | Force `alora` or `lora` when both exist for an adapter (default: prefer `alora`). |

The two cells below are pre-commented; uncomment one, change `--output`, and run it to see the effect.

**Example A - `--include-adapters`**: a lean checkpoint with only the intrinsics used in [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb) (guardian + 4 RAG adapters).

In [ ]:
# !python -m granite_switch.composer.compose_granite_switch \
#   --base-model {BASE_MODEL} \
#   --adapters {RAG_LIB} {GUARDIAN_LIB} \
#   --include-adapters guardian-core query_rewrite answerability \
#                      query_clarification citations \
#   --output ./granite-switch-hello-world

**Example B - `--exclude-adapters` + `--technology-filter`**: everything *except* the factuality adapters, using the LoRA flavor where both exist.

In [ ]:
# !python -m granite_switch.composer.compose_granite_switch \
#   --base-model {BASE_MODEL} \
#   --adapters {RAG_LIB} {CORE_LIB} {GUARDIAN_LIB} \
#   --exclude-adapters 'factuality-*' \
#   --technology-filter lora \
#   --output ./granite-switch-lora-no-factuality

## 5 * Serve the composed checkpoint

Start a vLLM server pointing at the directory section 3 produced:


In [ ]:
!python -m vllm.entrypoints.openai.api_server \
  --model ./granite-switch-tutorials \
  --port 8000



Leave this running in a separate terminal; section 6 below connects to it over HTTP.


## 6 * Generate against the composed model

Connect mellea to the running vLLM server, register the embedded adapters, and call the `rewrite_question` intrinsic. If it prints a cleaned-up version of the messy query, your composed checkpoint is wired up correctly.

In [ ]:
backend = OpenAIBackend(
    model_id=OUTPUT_DIR,
    base_url="http://localhost:8000/v1",
    api_key="unused",
)
backend.register_embedded_adapter_model(OUTPUT_DIR)
print(f"Adapters: {backend.list_adapters()}")

query = "I want to ask you something. what is...mmmm the the main city(capital you call it,right?) of France?"
rewritten = rag.rewrite_question(query, ChatContext(), backend)
print(f"\noriginal:  {query}")
print(f"rewritten: {rewritten}")

## 7 * Next steps

- **Compose with your own adapters.** The CLI accepts any HF repo id or local path with the adapter-library layout - swap `RAG_LIB` / `CORE_LIB` / `GUARDIAN_LIB` in section 1 for your own library to produce a task-specific checkpoint.
- **Consume the checkpoint.** Point [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb) or [`02_govt_rag_pipeline.ipynb`](02_govt_rag_pipeline.ipynb) at `./granite-switch-tutorials` to drive the intrinsics you just embedded.
- **HF-native inference.** [`01_granite_switch_with_hf.ipynb`](01_granite_switch_with_hf.ipynb) loads a composed checkpoint with `transformers` instead of vLLM - useful for router training or CPU prototyping.
- **Full CLI reference.** [`src/granite_switch/composer/README.md`](../../src/granite_switch/composer/README.md) documents every compose flag, including options not covered here.
- **Train your own adapter first.** [`../how-to/bring_your_own_adapter.md`](../how-to/bring_your_own_adapter.md) walks through producing a LoRA in the adapter-library layout this CLI consumes.